# Task 01 — Clean & Prepare a Dataset

**Dataset:** Titanic passenger manifest (891 passengers, Kaggle training set)
**Source:** https://github.com/datasciencedojo/datasets/blob/master/titanic.csv

**Goal:** load the raw data, find and fix quality problems (missing values, duplicates, incorrect or inconsistent values, wrong data types), export a clean file, and compare data quality before and after.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)

## 1. Load the dataset

In [2]:
raw = pd.read_csv("data/titanic_raw.csv")
df = raw.copy()
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Shape: 891 rows x 12 columns


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


A helper that measures data quality, so the same checks can be run before and after cleaning.

In [4]:
def quality_report(data):
    text_cols = data.select_dtypes(include=["object", "string", "str"]).columns
    return {
        "Rows": len(data),
        "Columns": data.shape[1],
        "Missing cells": int(data.isna().sum().sum()),
        "Duplicate rows (exact)": int(data.duplicated().sum()),
        "Duplicate rows (ignoring PassengerId)": int(data.drop(columns="PassengerId").duplicated().sum()),
        "Invalid fares (0)": int((data["Fare"] == 0).sum()),
        "Text with stray whitespace": int(sum((data[c].dropna() != data[c].dropna().str.strip()).sum() for c in text_cols)),
    }

before = quality_report(raw)
pd.Series(before, name="Before")

Rows                                     891
Columns                                   12
Missing cells                            866
Duplicate rows (exact)                     0
Duplicate rows (ignoring PassengerId)      0
Invalid fares (0)                         15
Text with stray whitespace                 2
Name: Before, dtype: int64

## 2. Identify missing values

In [5]:
missing = pd.DataFrame({"Missing": df.isna().sum(), "Percent": (df.isna().mean() * 100).round(1)})
missing[missing["Missing"] > 0]

,Missing,Percent
Age,177,19.9
Cabin,687,77.1
Embarked,2,0.2


- **Age**: 177 missing (19.9%) → impute.
- **Cabin**: 687 missing (77.1%) → too sparse to impute; keep the information as features instead.
- **Embarked**: 2 missing → fill with the most common port.

## 3. Remove duplicate records
Text is normalised first so rows that differ only in spacing or capitalisation are also caught.

In [6]:
for c in ["Name", "Ticket", "Cabin"]:
    df[c] = df[c].str.strip().str.replace(r"\s+", " ", regex=True)
df["Sex"] = df["Sex"].str.strip().str.lower()
df["Embarked"] = df["Embarked"].str.strip().str.upper()

n_before = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset=[c for c in df.columns if c != "PassengerId"])
print(f"Duplicates removed: {n_before - len(df)}")

Duplicates removed: 0


No duplicates were found, so no rows were removed. The check is kept in the pipeline so it protects against duplicates in future versions of the file.

## 4. Handle incorrect / inconsistent values

**4a. Titles.** Extract the title from each name and merge spelling variants (French *Mlle*/*Mme*, *Ms*) and rare honorifics.

In [7]:
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
print("Before:", df["Title"].value_counts().to_dict())

title_map = {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
             "Lady": "Rare", "the Countess": "Rare", "Capt": "Rare", "Col": "Rare", "Don": "Rare",
             "Dr": "Rare", "Major": "Rare", "Rev": "Rare", "Sir": "Rare", "Jonkheer": "Rare"}
df["Title"] = df["Title"].replace(title_map)
print("After: ", df["Title"].value_counts().to_dict())

Before: {'Mr': 517, 'Miss': 182, 'Mrs': 125, 'Master': 40, 'Dr': 7, 'Rev': 6, 'Major': 2, 'Mlle': 2, 'Col': 2, 'Don': 1, 'Mme': 1, 'Ms': 1, 'Lady': 1, 'Sir': 1, 'Capt': 1, 'the Countess': 1, 'Jonkheer': 1}
After:  {'Mr': 517, 'Miss': 185, 'Mrs': 126, 'Master': 40, 'Rare': 23}


**4b. Fares of 0.** A ticket price of £0 is not a real fare. These are flagged and treated as missing, then imputed in step 5.

In [8]:
df["FareWasZero"] = df["Fare"] == 0
print("Zero fares:", df["FareWasZero"].sum())
df.loc[df["FareWasZero"], ["Name", "Pclass", "Ticket", "Fare"]].head()

Zero fares: 15


,Name,Pclass,Ticket,Fare
179,"Leonard, Mr. Lionel",3,LINE,0.0
263,"Harrison, Mr. William",1,112059,0.0
271,"Tornquist, Mr. William Henry",3,LINE,0.0
277,"Parkes, Mr. Francis ""Frank""",2,239853,0.0
302,"Johnson, Mr. William Cahoone Jr",3,LINE,0.0


In [9]:
df.loc[df["FareWasZero"], "Fare"] = np.nan

**4c. Range and domain checks.** Ages must be between 0 and 100, family counts can't be negative, and categories must only contain valid values.

In [10]:
bad_age = df["Age"].notna() & ~df["Age"].between(0, 100)
bad_family = (df["SibSp"] < 0) | (df["Parch"] < 0)
print("Out-of-range ages:", bad_age.sum(), "| Negative family counts:", bad_family.sum())
df.loc[bad_age, "Age"] = np.nan

assert set(df["Sex"].unique()) <= {"male", "female"}
assert set(df["Pclass"].unique()) <= {1, 2, 3}
assert set(df["Survived"].unique()) <= {0, 1}
print("Category domains are valid.")

Out-of-range ages: 0 | Negative family counts: 0
Category domains are valid.


## 5. Handle missing values

**Age:** median by Title and Pclass. A global median (28) would give boys titled *Master* an adult age.

In [11]:
df.groupby(["Title", "Pclass"])["Age"].median().unstack()

Pclass,1,2,3
Title,,,
Master,4.0,1.0,4.0
Miss,30.0,24.0,18.0
Mr,40.0,31.0,26.0
Mrs,40.0,32.0,31.0
Rare,48.5,46.5,NaN


In [12]:
df["AgeWasMissing"] = df["Age"].isna()
df["Age"] = df["Age"].fillna(df.groupby(["Title", "Pclass"])["Age"].transform("median"))
df["Age"] = df["Age"].fillna(df["Age"].median())

df["Fare"] = df["Fare"].fillna(df.groupby("Pclass")["Fare"].transform("median")).round(4)

print("Missing Embarked rows:")
display(df.loc[df["Embarked"].isna(), ["Name", "Ticket", "Fare", "Cabin"]])
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

df["HasCabin"] = df["Cabin"].notna()
df["Deck"] = df["Cabin"].str[0].fillna("Unknown")
df["Cabin"] = df["Cabin"].fillna("Unknown")

print("Missing values left:", df.isna().sum().sum())

Missing Embarked rows:


,Name,Ticket,Fare,Cabin
61,"Icard, Miss. Amelie",113572,80.0,B28
829,"Stone, Mrs. George Nelson (Martha Evelyn)",113572,80.0,B28


Missing values left: 0


## 6. Convert columns to appropriate data types

In [13]:
dtypes_before = df.dtypes.astype(str)
df = df.astype({
    "PassengerId": "int32",
    "Survived": "bool",
    "Pclass": pd.CategoricalDtype([1, 2, 3], ordered=True),
    "Sex": "category", "Embarked": "category", "Title": "category", "Deck": "category",
    "SibSp": "int8", "Parch": "int8",
    "Age": "float32", "Fare": "float64",
    "Name": "string", "Ticket": "string", "Cabin": "string",
})
pd.DataFrame({"Before": dtypes_before, "After": df.dtypes.astype(str)})

,Before,After
PassengerId,int64,int32
Survived,int64,bool
Pclass,int64,category
Name,str,string
Sex,str,category
Age,float64,float32
SibSp,int64,int8
Parch,int64,int8
Ticket,str,string
Fare,float64,float64


In [14]:
mem_before = raw.memory_usage(deep=True).sum() / 1024
mem_after = df.memory_usage(deep=True).sum() / 1024
print(f"Memory: {mem_before:.0f} KB -> {mem_after:.0f} KB")

Memory: 286 KB -> 187 KB


## 7. Save the cleaned dataset

In [15]:
df.to_csv("data/titanic_clean.csv", index=False)
print("Saved data/titanic_clean.csv:", df.shape)

Saved data/titanic_clean.csv: (891, 17)


## 8. Before / after data quality

In [16]:
after = quality_report(df)
comparison = pd.DataFrame({"Before": before, "After": after})
comparison

,Before,After
Rows,891,891
Columns,12,17
Missing cells,866,0
Duplicate rows (exact),0,0
Duplicate rows (ignoring PassengerId),0,0
Invalid fares (0),15,0
Text with stray whitespace,2,0


In [17]:
pd.DataFrame({"Missing before": raw.isna().sum(), "Missing after": df[raw.columns].isna().sum()}).query("`Missing before` > 0")

,Missing before,Missing after
Age,177,0
Cabin,687,0
Embarked,2,0


In [18]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,FareWasZero,AgeWasMissing,HasCabin,Deck
0,1,False,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,Unknown,S,Mr,False,False,False,Unknown
1,2,True,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Mrs,False,False,True,C
2,3,True,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,Unknown,S,Miss,False,False,False,Unknown
3,4,True,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Mrs,False,False,True,C
4,5,False,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,Unknown,S,Mr,False,False,False,Unknown


## Summary

| Problem | Fix |
|---|---|
| 177 missing ages | Median by Title × Pclass, flagged in `AgeWasMissing` |
| 687 missing cabins | Labelled `Unknown`; `HasCabin` and `Deck` derived |
| 2 missing ports | Filled with the most common port, `S` (Southampton) |
| 15 fares of £0 | Treated as invalid, replaced with the class median, flagged in `FareWasZero` |
| Inconsistent titles | Mlle/Ms → Miss, Mme → Mrs, rare honorifics → Rare |
| Duplicates | Checked (exact and ignoring ID); none found |
| Generic types | bool, ordered category, category, int8, string |

The cleaned file has **0 missing values, 0 duplicates and 0 invalid fares**, and is ready for analysis or modelling.